In [15]:
import os
from dotenv import load_dotenv, find_dotenv

_ = load_dotenv(find_dotenv())
openai_api_key = os.getenv("OPENAI_API_KEY")


In [17]:
from langchain_openai import ChatOpenAI

chatModel = ChatOpenAI(model="gpt-3.5-turbo-0125")

In [11]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda

def russian_lasname(name: str) -> str:
    return name + "ovich"

chain = RunnableParallel(
    {
        'operation_a':RunnablePassthrough(),
        'operation_b':RunnableLambda(russian_lasname)
    }
)

In [12]:
chain.invoke("Ange")

{'operation_a': 'Ange', 'operation_b': 'Angeovich'}

In [24]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("Tell me a curious fact about {basketball_player}")

output_parser = StrOutputParser()

def russian_lastname_from_dictionary(person) -> str:
    return person["name"] + "ovich"



In [ ]:
chain = RunnableParallel(
    {
        'operation_a':RunnablePassthrough(),
        'basketball_player':RunnableLambda(russian_lastname_from_dictionary),
        'operation_c':RunnablePassthrough()
    }
) | prompt | chatModel | output_parser

In [34]:
chain.invoke({
    'nickname': 'The Brain',
    'name': 'Pop',
})

'Gregg Popovich, the head coach of the San Antonio Spurs, is known for his dry sense of humor and sarcastic comments during interviews. Many fans and journalists find his quirkiness and wit entertaining, adding to his reputation as one of the most respected and beloved coaches in the NBA.'

### Bult-in functions in runnables

In [39]:
chain = RunnableParallel(
    {
        'operation_a':RunnablePassthrough(),
        'basketball_player':RunnableLambda(russian_lastname_from_dictionary),
        'operation_c':RunnablePassthrough()
    }
) | prompt | chatModel.bind(stop=["Popovich "]) | output_parser

In [40]:
chain.invoke({
    'nickname': 'The Brain',
    'name': 'Pop',
})

'One curious fact about Gregg '

### Combining chains

In [ ]:
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

prompt1 = ChatPromptTemplate.from_template("what is the country {politician} is from?")
prompt2 = ChatPromptTemplate.from_template(
    "what continent is the country {country} in? respond in {language}"
)

model = ChatOpenAI()

chain1 = prompt1 | model | StrOutputParser()

chain2 = (
    {"country": chain1, "language": itemgetter("language")}
    | prompt2
    | model
    | StrOutputParser() # parse the final output as a string: response.content
)

chain2.invoke({"politician": "Miterrand", "language": "French"})

"Mitterrand est originaire d'Europe, plus précisément du continent européen."

### LCEL chain at work in a typical RAG app

In [45]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-3.5-turbo-0125", temperature=0)

In [ ]:
import bs4
from langchain_chroma import Chroma
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)

docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

splits = text_splitter.split_documents(docs)

vectorstore = Chroma.from_documents(documents=splits, embedding=OpenAIEmbeddings())
"""vectorstore = FAISS.from_texts(

    ["cats like fish", "dogs like sticks"],

    embedding=OpenAIEmbeddings()

)"""
retriever = vectorstore.as_retriever()

prompt = ChatPromptTemplate.from_template(
    "Answer the question using only the context below. "
    "If you do not know the answer, say you do not know. Keep the answer concise.\n\n"
    "Context:\n{context}\n\nQuestion: {question}\n\nAnswer:"
)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | model
    | StrOutputParser()
)

In [52]:
rag_chain.invoke("What is Task Decomposition?")

'Task Decomposition is a technique that breaks down complex tasks into smaller and simpler steps to enhance model performance.'